# Apache Hudi Quickstart — PySpark Edition

This notebook mirrors every example from `hudi_pipeline_quickstart.scala` using PySpark.
The Hudi DataSource API (`.write.format("hudi")`, `.read.format("hudi")`, and all `.option()` calls)
is identical across Scala and Python — only the DataFrame creation syntax differs.

**Requirements:** Spark 3.5.x with the Hudi 1.2.0 bundle JAR on the classpath.
If using the Docker setup, everything is pre-configured — just run `docker compose up`.

## Configuration

In [ ]:
# Update these paths for your environment.
# If running inside the Docker container, the default paths work as-is.
input_path = "/home/hudi/data/trips_0"   # Path to extracted trips_0 file
base_path  = "/tmp/trips_table"           # CoW table location
mor_base_path = "/tmp/trips_table_mor"    # MoR table location

## Section 1: Data Loading and CoW Table Creation

In [ ]:
# Load the NYC taxi dataset from tab-separated CSV.
# inferSchema samples the file to detect column types (fare_amount as double, etc.)
# instead of reading everything as strings.
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", "\t") \
    .option("inferSchema", "true") \
    .load(input_path)

df.printSchema()
print(f"Loaded {df.count()} rows")

In [ ]:
# Create a Copy-on-Write (CoW) Hudi table.
# CoW rewrites entire base files on every update — optimized for read-heavy workloads.
df.write.format("hudi") \
    .option("hoodie.datasource.write.recordkey.field", "trip_id") \
    .option("hoodie.datasource.write.partitionpath.field", "vendor_id") \
    .option("hoodie.datasource.write.hive_style_partitioning", "true") \
    .option("hoodie.table.name", "nyc_taxi_trips") \
    .mode("Overwrite") \
    .save(base_path)

## Section 2: Table Verification and Upsert Operations

In [ ]:
# Verify table creation with a snapshot query.
snapshot_df = spark.read.format("hudi").load(base_path)
print(f"Row count: {snapshot_df.count()}")

snapshot_df.select("trip_id", "vendor_id", "pickup_datetime", "fare_amount") \
    .filter("vendor_id = '1'").limit(2).show()

In [ ]:
from pyspark.sql.functions import col

# Update an existing record: increase fare_amount by 20%.
to_upsert = df.filter(col("trip_id") == "1207977523") \
    .withColumn("fare_amount", col("fare_amount").cast("double") * 1.2)

# Perform the upsert. Use mode("Append") — not "Overwrite" — to add to the table.
to_upsert.write.format("hudi") \
    .option("hoodie.datasource.write.operation", "upsert") \
    .mode("Append") \
    .save(base_path)

In [ ]:
# Verify: fare_amount should now be 19.2 (16 * 1.2).
snapshot_df2 = spark.read.format("hudi").load(base_path)
snapshot_df2.select("trip_id", "vendor_id", "pickup_datetime", "fare_amount") \
    .filter(col("trip_id") == "1207977523").show()

## Section 3: Delete Operations

In [ ]:
# Delete records with very short trip distances (< 0.1 miles).
to_delete = df.filter(col("trip_distance").cast("double") < 0.1)
print(f"Records to delete: {to_delete.count()}")

to_delete.select("trip_id", "vendor_id", "pickup_datetime", "trip_distance") \
    .filter("vendor_id = '1'").limit(1).show()

# Perform a hard delete.
to_delete.write.format("hudi") \
    .option("hoodie.datasource.write.operation", "delete") \
    .mode("Append") \
    .save(base_path)

In [ ]:
# Verify: total count should decrease and the deleted record should not appear.
snapshot_df3 = spark.read.format("hudi").load(base_path)
print(f"Row count after delete: {snapshot_df3.count()}")

snapshot_df3.select("trip_id", "vendor_id", "pickup_datetime", "fare_amount") \
    .filter(col("trip_id") == "1200001601").show()
# Expected: empty result — the record was deleted.

## Section 4: Exploring Hudi Metadata and Commit Timeline

Hudi's metadata APIs are Java/Scala. We use `spark._jvm` to access them from PySpark.

In [ ]:
# Access Hudi's Java APIs via spark._jvm.
jvm = spark._jvm

hadoop_conf = spark._jsc.hadoopConfiguration()
storage_conf = jvm.org.apache.hudi.storage.hadoop.HadoopStorageConfiguration(hadoop_conf)

meta = jvm.org.apache.hudi.common.table.HoodieTableMetaClient.builder() \
    .setBasePath(base_path) \
    .setConf(storage_conf) \
    .build()

# Retrieve completed instants.
# Each instant has two timestamps:
#   requestedTime — when the action was requested (used for ordering and as-of queries)
#   completionTime — when the action finished (used for incremental query bounds)
timeline = meta.getCommitsTimeline().filterCompletedInstants()
instants_iter = timeline.getInstants().iterator()

instants = []
while instants_iter.hasNext():
    instants.append(instants_iter.next())

instants.sort(key=lambda i: i.requestedTime())

for i in instants:
    print(f"  requested={i.requestedTime()}  completed={i.getCompletionTime()}  action={i.getAction()}")

first_commit = instants[0]
upsert_commit = instants[1]
delete_commit = instants[2]

## Section 5: Hudi Query Types

In [ ]:
# Read-optimized query.
# For CoW: identical to snapshot (no log files exist).
# For MoR: returns only compacted base files — faster but may lag behind the latest writes.
ro_df = spark.read.format("hudi") \
    .option("hoodie.datasource.query.type", "read_optimized") \
    .load(base_path)
print(f"Read-optimized count: {ro_df.count()}")

In [ ]:
# Incremental query — returns changes between two completion-time bounds.
# Boundary semantics (Hudi 1.x, completion-time mode):
#   begin is exclusive (>), end is inclusive (<=).
incr_df = spark.read.format("hudi") \
    .option("hoodie.datasource.query.type", "incremental") \
    .option("hoodie.datasource.read.begin.instanttime", first_commit.getCompletionTime()) \
    .option("hoodie.datasource.read.end.instanttime", delete_commit.getCompletionTime()) \
    .load(base_path)

incr_df.select("trip_id", "vendor_id", "pickup_datetime", "fare_amount").show(truncate=False)
# Returns the latest state of keys affected between first_commit and delete_commit.
# NOTE: This is a latest-state result, not a before/after change log.
# For full CDC semantics, enable CDC on the table and use the CDC query format.

In [ ]:
# Time-travel query — read the table as it existed at a past instant.
# Use requestedTime for the as-of bound.
tt_df = spark.read.format("hudi") \
    .option("as.of.instant", upsert_commit.requestedTime()) \
    .load(base_path)
print(f"Time-travel count: {tt_df.count()}")

# The deleted record should still exist at this point in time.
tt_df.select("trip_id", "vendor_id", "pickup_datetime", "fare_amount") \
    .filter(col("trip_id") == "1200001601").show()

## Section 6: Merge-on-Read (MoR) Table Operations

In [ ]:
# Create a MoR table. Updates go to append-only log files instead of rewriting
# base files — much faster writes, but snapshot reads must merge logs at read time.
df.write.format("hudi") \
    .option("hoodie.datasource.write.recordkey.field", "trip_id") \
    .option("hoodie.datasource.write.partitionpath.field", "vendor_id") \
    .option("hoodie.datasource.write.hive_style_partitioning", "true") \
    .option("hoodie.datasource.write.table.type", "MERGE_ON_READ") \
    .option("hoodie.table.name", "nyc_taxi_trips_mor") \
    .mode("Overwrite") \
    .save(mor_base_path)

In [ ]:
# Upsert on MoR — writes go to log files, no base-file rewrite.
to_upsert.write.format("hudi") \
    .option("hoodie.datasource.write.table.type", "MERGE_ON_READ") \
    .option("hoodie.datasource.write.operation", "upsert") \
    .mode("Append") \
    .save(mor_base_path)

# Delete on MoR — delete markers go to log files.
to_delete.write.format("hudi") \
    .option("hoodie.datasource.write.table.type", "MERGE_ON_READ") \
    .option("hoodie.datasource.write.operation", "delete") \
    .mode("Append") \
    .save(mor_base_path)

In [ ]:
# Snapshot query on MoR — merges base files + log files for the complete view.
mor_snapshot = spark.read.format("hudi").load(mor_base_path)
print(f"MoR snapshot count: {mor_snapshot.count()}")

# Read-optimized query on MoR — reads only compacted base files, skips logs.
# Count will differ from snapshot because updates/deletes in logs are not visible.
mor_ro = spark.read.format("hudi") \
    .option("hoodie.datasource.query.type", "read_optimized") \
    .load(mor_base_path)
print(f"MoR read-optimized count: {mor_ro.count()}")

## Section 7: Table Maintenance Operations

In [ ]:
# --- Compaction ---
# Compaction merges the current base file with its eligible log files, producing
# a new base-file version. It does NOT rewrite every historical slice.
# Inline compaction runs on the writer path and adds write latency.
to_upsert.write.format("hudi") \
    .option("hoodie.datasource.write.table.type", "MERGE_ON_READ") \
    .option("hoodie.datasource.write.operation", "upsert") \
    .option("hoodie.compact.inline", "true") \
    .option("hoodie.compact.inline.max.delta.commits", "1") \
    .mode("Append") \
    .save(mor_base_path)

# After compaction, snapshot and read-optimized return the same logical rows.
mor_snap2 = spark.read.format("hudi").load(mor_base_path)
mor_ro2 = spark.read.format("hudi") \
    .option("hoodie.datasource.query.type", "read_optimized") \
    .load(mor_base_path)
print(f"After compaction — snapshot: {mor_snap2.count()}, read-optimized: {mor_ro2.count()}")

In [ ]:
# --- Clustering ---
# Clustering reorganizes files: combines small files into larger ones and
# optionally sorts by columns for better data skipping during queries.
df.limit(0).write.format("hudi") \
    .option("hoodie.datasource.write.operation", "upsert") \
    .option("hoodie.datasource.write.table.type", "MERGE_ON_READ") \
    .option("hoodie.clustering.inline", "true") \
    .option("hoodie.clustering.inline.max.commits", "1") \
    .option("hoodie.clustering.plan.strategy.small.file.limit", "10485760") \
    .option("hoodie.clustering.plan.strategy.target.file.max.bytes", "41943040") \
    .option("hoodie.clustering.plan.strategy.sort.columns", "pickup_date") \
    .mode("Append") \
    .save(mor_base_path)

In [ ]:
# --- Cleaning ---
# Cleaning removes older file-slice versions that fall outside the retention policy.
# These older slices are valid MVCC history (time travel, rollback, reader isolation)
# until the retention window expires.
#
# Default behavior: cleaning runs inline with the writer automatically.
# Compaction and clustering do NOT run inline by default.
df.limit(0).write.format("hudi") \
    .option("hoodie.datasource.write.operation", "upsert") \
    .option("hoodie.clean.commits.retained", "1") \
    .option("hoodie.clean.automatic", "true") \
    .mode("Append") \
    .save(mor_base_path)

print("Cleaning complete. Only the latest file-slice version is retained per file group.")